In [1]:
from langgraph.store.memory import InMemoryStore

# ==================== 1. 创建Store ====================
store = InMemoryStore()

In [4]:
from dataclasses import dataclass


@dataclass(frozen=True)
class UserContext:
    """Agent运行时上下文"""
    user_id: str
    tenant_id: str

USER_DATABASE = {
    "school_a": {
        "u001": {"name": "虎哥", "course": "LangChain 开发指南"},
        "u002": {"name": "小明", "course": "Java开发入门"},
    },
    "school_b": {
        "u001": {"name": "涛哥", "course": "Python 入门"},
        "u002": {"name": "阿伟", "course": "VibeCoding 实战"},
    }
}

In [5]:
from langchain.tools import ToolRuntime, tool


# 定义方法，根据Context中的tenant_id和user_id拼接得到namespace
def preference_namespace(ctx: UserContext) -> tuple[str, ...]:
    return (
        "heima_platform",
        ctx.tenant_id,
        ctx.user_id
    )
# 定义固定的key
key = "learning_preferences"


@tool
def get_current_user_profile(runtime: ToolRuntime[UserContext]):
    """查询当前登录用户的资料。"""

    # 获取context
    ctx = runtime.context

    # 根据user_id和tenant_id获取用户信息
    profile = USER_DATABASE.get(ctx.tenant_id, {}).get(ctx.user_id)

    return (
        f"姓名：{profile['name']}，"
        f"课程：{profile['course']}，"
    )

@tool
def save_learning_preference(
    preference_name: str,
    preference_value: str,
    runtime: ToolRuntime[UserContext],
) -> str:
    """保存当前用户的长期学习偏好。"""

    # 通过上下文中的tenant_id和user_id来构建namespace
    namespace = preference_namespace(runtime.context)

    # 通过namespace和key来读取用户偏好, 返回结果item.value是dict格式
    item = runtime.store.get(namespace, key)
    preferences = dict(item.value) if item else {}

    # 记录本次要保存的用户偏好
    preferences[preference_name] = preference_value

    # 保存到Store中
    runtime.store.put(namespace, key, preferences)

    return f"已保存偏好：{preference_name}={preference_value}"


@tool
def get_learning_preferences(
    runtime: ToolRuntime[UserContext],
) -> str:
    """读取当前用户的长期学习偏好。"""

    namespace = preference_namespace(runtime.context)
    item = runtime.store.get(namespace, key)

    if item is None:
        return "当前用户还没有保存学习偏好"

    return f"当前学习偏好：{item.value}"

In [6]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

from dotenv import load_dotenv

load_dotenv()


checkpointer = InMemorySaver()
store = InMemoryStore()

agent = create_agent(
    model="deepseek-flash",
    tools=[
        get_current_user_profile,
        save_learning_preference,
        get_learning_preferences,
    ],
    context_schema=UserContext,
    checkpointer=checkpointer,
    store=store,
    system_prompt=(
        "你是课程平台助手。"
        "涉及用户资料时调用资料工具，"
        "涉及长期学习偏好或给用户讲解代码时调用偏好工具。"
    ),
)

In [7]:
context = UserContext(
    user_id="u001",
    tenant_id="school_a",
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "记住：我喜欢先看代码，再看原理",
            }
        ]
    },
    config={"configurable": {"thread_id": "thread-a"}},
    context=context,
)
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

记住：我喜欢先看代码，再看原理
================================== Ai Message ==================================
Tool Calls:
  save_learning_preference (call_00_vqXTwUxZU0FzUmm8V2Kz6924)
 Call ID: call_00_vqXTwUxZU0FzUmm8V2Kz6924
  Args:
    preference_name: 讲解顺序偏好
    preference_value: 先看代码示例，再看原理讲解（代码优先）
================================= Tool Message =================================
Name: save_learning_preference

已保存偏好：讲解顺序偏好=先看代码示例，再看原理讲解（代码优先）
================================== Ai Message ==================================

已记住 ✅

**偏好已保存：讲解顺序偏好 = 先看代码示例，再看原理讲解（代码优先）**

之后我讲解知识点时，会按这个顺序来安排：

1. **先给可运行的代码示例** —— 你可以先直接看、直接跑，从具体实现入手
2. **再讲背后的原理** —— 为什么这么写、机制是什么、有什么坑和替代方案

如果你想临时调整某次讲解的顺序，或者想再加一条偏好（比如「代码要带注释」「原理部分尽量画图/列对比表」），直接告诉我就行。现在有想让我讲解的内容吗？可以发题目、代码或知识点给我。


In [8]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "langchain如何实现结构化输出？",
            }
        ]
    },
    config={"configurable": {"thread_id": "thread-b"}},
    context=context,
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

langchain如何实现结构化输出？
================================== Ai Message ==================================

我先查一下你的资料和学习偏好，好按你的习惯来讲解。
Tool Calls:
  get_current_user_profile (call_00_Javls79LfuxmdkEsxZan3668)
 Call ID: call_00_Javls79LfuxmdkEsxZan3668
  Args:
  get_learning_preferences (call_01_tN0rNl9G2XSHR90DcefL9277)
 Call ID: call_01_tN0rNl9G2XSHR90DcefL9277
  Args:
================================= Tool Message =================================
Name: get_current_user_profile

姓名：虎哥，课程：LangChain 开发指南，
================================= Tool Message =================================
Name: get_learning_preferences

当前学习偏好：{'讲解顺序偏好': '先看代码示例，再看原理讲解（代码优先）'}
================================== Ai Message ==================================

虎哥，按你的习惯先看代码，原理放后面。

## 1. 最简用法：`with_structured_output`

LangChain 里最直接的入口是 chat model 上的 `with_structured_output()`：

```python
from pydantic import BaseModel, Field
from typin